# Identifying policy value under overlap

## Building an action log from public basket returns

The public archive supplies the two returns that could be received on every date. The defensive basket is the equal weighted return of Food, Smoke, Health, and Utilities. The cyclical basket is the equal weighted return of Building Materials, Steel, Machinery, and Automobiles. The action log is generated because the archive contains returns but no historical record of a decision maker choosing between these baskets.

Let $Y_i(0)$ and $Y_i(1)$ be the two public basket returns on date $i$. The realized reward is

$$
Y_i=Y_i(A_i),
$$

where $A_i=0$ selects the defensive basket and $A_i=1$ selects the cyclical basket. The context $X_i$ contains a twenty day mean return spread, a twenty day volatility measure, and the previous day's spread. Every context value is formed before date $i$'s reward.

In [1]:
from pathlib import Path
import hashlib
import io
import sys
import zipfile

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)


BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset
FINANCIAL_ROOT = BOOK_ROOT

import RiskLabAI
from RiskLabAI.causal_factor_analysis import evaluate_policy, select_policy

assert RiskLabAI.__version__ == "3.2.0"

FIGURES = BOOK_ROOT / "outputs" / "figures" / "09_policy_evaluation"
TABLES = BOOK_ROOT / "outputs" / "tables" / "09_policy_evaluation"
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "09_policy_evaluation"
DATA_FILE = ensure_dataset(BOOK_ROOT, "french_49_industries")
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "violet": "#A03A78",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 11.5,
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["axes"],
    "axes.edgecolor": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "text.color": PALETTE["text"],
    "grid.color": PALETTE["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": PALETTE["axes"],
    "legend.edgecolor": PALETTE["grid"],
    "savefig.facecolor": PALETTE["background"],
    "savefig.transparent": False,
    "savefig.dpi": 300,
})


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (bool, np.bool_)):
        return "Pass" if bool(value) else "Fail"
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        value = float(value)
        if value == 0:
            return "0"
        if value.is_integer():
            return f"{int(value):,}"
        if abs(value) < 0.0001 or abs(value) >= 1000000:
            return f"{value:.6g}"
        return f"{value:.6f}".rstrip("0").rstrip(".")
    return str(value)


def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    latex = latex.replace("\\begin{table}[htbp]", "\\begin{table}[htbp]\n\\centering", 1)
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def finish_axis(axis, grid=True):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)


def save_figure(figure, stem):
    figure.savefig(FIGURES / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    figure.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    display(figure)
    plt.close(figure)


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def read_public_returns():
    with zipfile.ZipFile(DATA_FILE) as archive:
        lines = archive.read(archive.namelist()[0]).decode("latin1").splitlines()
    header = next(index for index, line in enumerate(lines) if line.startswith(",Agric"))
    ending = next(index for index in range(header + 1, len(lines)) if not lines[index].strip())
    raw = pd.read_csv(io.StringIO("\n".join(lines[header:ending])))
    raw = raw.rename(columns={raw.columns[0]: "Date"})
    raw["Date"] = pd.to_datetime(raw["Date"].astype(str), format="%Y%m%d")
    return raw.set_index("Date").replace([-99.99, -999.0], np.nan) / 100


def logistic(values):
    clipped = np.clip(values, -35, 35)
    return 1 / (1 + np.exp(-clipped))


def build_public_frame(floor=0.10, random_state=9101):
    returns = read_public_returns()
    defensive = ["Food", "Smoke", "Hlth", "Util"]
    cyclical = ["BldMt", "Steel", "Mach", "Autos"]
    baskets = pd.DataFrame({
        "Defensive return": returns[defensive].mean(axis=1),
        "Cyclical return": returns[cyclical].mean(axis=1),
    })
    spread = baskets["Cyclical return"] - baskets["Defensive return"]
    market = baskets.mean(axis=1)
    context = pd.DataFrame({
        "Past mean spread": spread.rolling(20).mean().shift(1),
        "Past volatility": market.rolling(20).std(ddof=1).shift(1),
        "Past one-day spread": spread.shift(1),
    })
    reference = context.loc["2018-01-01":"2018-12-31"].dropna()
    center = reference.mean()
    scale = reference.std(ddof=1)
    standardized = (context - center) / scale
    frame = pd.concat([baskets, context, standardized.add_suffix(" z")], axis=1).loc["2019-01-01":"2023-12-31"].dropna().copy()
    score = 0.70 * frame["Past mean spread z"] - 0.40 * frame["Past volatility z"] + 0.20 * frame["Past one-day spread z"]
    probability_one = floor + (1 - 2 * floor) * logistic(score)
    uniforms = np.random.default_rng(random_state).random(len(frame))
    actions = (uniforms < probability_one.to_numpy()).astype(int)
    potential = frame[["Defensive return", "Cyclical return"]].to_numpy()
    frame["Behavior probability action 1"] = probability_one
    frame["Behavior probability action 0"] = 1 - probability_one
    frame["Action"] = actions
    frame["Observed reward"] = potential[np.arange(len(frame)), actions]
    frame["Uniform draw"] = uniforms
    return frame


def behavior_matrix(frame):
    return frame[["Behavior probability action 0", "Behavior probability action 1"]].to_numpy()


def potential_matrix(frame):
    return frame[["Defensive return", "Cyclical return"]].to_numpy()


def candidate_policies(frame):
    spread = frame["Past mean spread z"].to_numpy()
    volatility = frame["Past volatility z"].to_numpy()
    probabilities_one = [
        np.zeros(len(frame)),
        np.ones(len(frame)),
        (spread > 0).astype(float),
        logistic(1.50 * spread),
        logistic(1.20 * spread - 0.60 * volatility),
    ]
    candidates = np.stack([np.column_stack([1 - values, values]) for values in probabilities_one])
    names = ["Always defensive", "Always cyclical", "Positive spread threshold", "Smooth momentum", "Volatility aware rule"]
    return names, candidates


def full_information_value(frame, policy, indices=None):
    rows = np.arange(len(frame)) if indices is None else np.asarray(indices)
    return float(np.mean(np.sum(policy[rows] * potential_matrix(frame)[rows], axis=1)))


def design_matrix(frame):
    return np.column_stack([
        np.ones(len(frame)),
        frame["Past mean spread z"].to_numpy(),
        frame["Past volatility z"].to_numpy(),
        frame["Past one-day spread z"].to_numpy(),
    ])


def fitted_coefficients(frame, training_rows):
    features = design_matrix(frame)
    actions = frame["Action"].to_numpy()
    rewards = frame["Observed reward"].to_numpy()
    coefficients = []
    for action in [0, 1]:
        rows = np.asarray(training_rows)[actions[np.asarray(training_rows)] == action]
        coefficients.append(np.linalg.lstsq(features[rows], rewards[rows], rcond=None)[0])
    return np.asarray(coefficients)


def chronological_predictions(frame):
    boundaries = [200, 370, 540, 710, 880, len(frame)]
    features = design_matrix(frame)
    predictions = np.zeros((len(frame), 2))
    folds = []
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        coefficients = fitted_coefficients(frame, np.arange(start))
        predictions[start:end] = features[start:end] @ coefficients.T
        folds.append({
            "Training first date": str(frame.index[0].date()),
            "Training last date": str(frame.index[start - 1].date()),
            "Prediction first date": str(frame.index[start].date()),
            "Prediction last date": str(frame.index[end - 1].date()),
            "Training rows": start,
            "Prediction rows": end - start,
            "Training precedes prediction": bool(frame.index[start - 1] < frame.index[start]),
        })
    return predictions, pd.DataFrame(folds)


def capture_failure(name, calculation):
    try:
        calculation()
        return {"Check": name, "Rejected": False, "Explanation": "Calculation continued"}
    except ValueError:
        return {"Check": name, "Rejected": True, "Explanation": "Stopped as required"}

frame = build_public_frame()
names, candidates = candidate_policies(frame)
behavior = behavior_matrix(frame)
potential = potential_matrix(frame)
actions = frame["Action"].to_numpy()
rewards = frame["Observed reward"].to_numpy()
predictions, fold_plan = chronological_predictions(frame)

provenance = pd.DataFrame([
    {"Item": "Public source", "Value": "Kenneth R. French 49 Industry Portfolios"},
    {"Item": "Stored archive", "Value": DATA_FILE.name},
    {"Item": "Archive bytes", "Value": DATA_FILE.stat().st_size},
    {"Item": "SHA256", "Value": file_sha256(DATA_FILE)},
    {"Item": "Analysis start", "Value": str(frame.index.min().date())},
    {"Item": "Analysis end", "Value": str(frame.index.max().date())},
    {"Item": "Complete dates", "Value": len(frame)},
    {"Item": "Download during execution", "Value": "None"},
])
assert file_sha256(DATA_FILE) == "8F394FE34BEA54D41B9AAFED410425EE8F8E252EDE3C71A7C1CD20BAB83040DE"
assert len(frame) == 1258
save_table(provenance, "ipw_data_provenance", "Provenance of the public returns used in the policy illustration.", "tab:cim9-ipw-data-provenance", fit=True)

basket_definitions = pd.DataFrame([
    {"Action": 0, "Basket": "Defensive", "Industries": "Food, Smoke, Health, Utilities", "Daily return rule": "Equal weighted"},
    {"Action": 1, "Basket": "Cyclical", "Industries": "Building Materials, Steel, Machinery, Automobiles", "Daily return rule": "Equal weighted"},
])
save_table(basket_definitions, "ipw_basket_definitions", "Fixed public basket definitions.", "tab:cim9-ipw-basket-definitions", fit=True)

sample_contract = pd.DataFrame([
    {"Component": "Potential rewards", "Source or rule": "Both public basket returns on the same date"},
    {"Component": "Context", "Source or rule": "Past returns only"},
    {"Component": "Behavior action", "Source or rule": "Generated from a declared probability and random state 9,101"},
    {"Component": "Behavior probability floor", "Source or rule": 0.10},
    {"Component": "Target policies", "Source or rule": "Five fixed rules declared before held-out evaluation"},
])
save_table(sample_contract, "ipw_sample_contract", "Data and decision contract for the illustration.", "tab:cim9-ipw-sample-contract", fit=True)

context_summary = frame[["Past mean spread", "Past volatility", "Past one-day spread"]].describe().T.reset_index().rename(columns={"index": "Context", "count": "Dates", "mean": "Mean", "std": "Standard deviation", "min": "Minimum", "max": "Maximum"})[["Context", "Dates", "Mean", "Standard deviation", "Minimum", "Maximum"]]
save_table(context_summary, "ipw_context_summary", "Summary of the past-only context variables.", "tab:cim9-ipw-context-summary", fit=True)

behavior_summary = pd.DataFrame([
    {"Quantity": "Minimum action probability", "Value": behavior.min()},
    {"Quantity": "Maximum action probability", "Value": behavior.max()},
    {"Quantity": "Mean probability of action 1", "Value": behavior[:, 1].mean()},
    {"Quantity": "Realized share of action 1", "Value": actions.mean()},
    {"Quantity": "Action 0 dates", "Value": int((actions == 0).sum())},
    {"Quantity": "Action 1 dates", "Value": int((actions == 1).sum())},
])
save_table(behavior_summary, "ipw_behavior_summary", "Summary of the generated behavior policy and realized actions.", "tab:cim9-ipw-behavior-summary")

candidate_definitions = pd.DataFrame([
    {"Policy": names[0], "Probability of action 1": "0"},
    {"Policy": names[1], "Probability of action 1": "1"},
    {"Policy": names[2], "Probability of action 1": "One when the past mean spread is positive"},
    {"Policy": names[3], "Probability of action 1": "Logistic function of 1.5 times the past mean spread"},
    {"Policy": names[4], "Probability of action 1": "Logistic function of past spread and volatility"},
])
save_table(candidate_definitions, "ipw_candidate_definitions", "Five fixed target policies.", "tab:cim9-ipw-candidate-definitions", fit=True)

cumulative = (1 + frame[["Defensive return", "Cyclical return"]]).cumprod() - 1
cumulative_data = cumulative.reset_index()
cumulative_data.to_csv(FIGURE_DATA / "ipw_basket_cumulative.csv", index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.4), constrained_layout=True)
axis.plot(cumulative.index, cumulative["Defensive return"], color=PALETTE["cyan"], linewidth=1.8, label="Defensive basket")
axis.plot(cumulative.index, cumulative["Cyclical return"], color=PALETTE["amber"], linewidth=1.8, label="Cyclical basket")
axis.set_ylabel("Cumulative return")
axis.set_title("Public Basket Returns")
axis.legend()
finish_axis(axis)
save_figure(figure, "ipw_basket_cumulative")

context_data = frame[["Past mean spread z", "Past volatility z"]].reset_index()
context_data.to_csv(FIGURE_DATA / "ipw_context_paths.csv", index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.4), constrained_layout=True)
axis.plot(frame.index, frame["Past mean spread z"], color=PALETTE["cyan"], linewidth=1.2, label="Past mean spread")
axis.plot(frame.index, frame["Past volatility z"], color=PALETTE["amber"], linewidth=1.2, label="Past volatility")
axis.axhline(0, color=PALETTE["grid"], linewidth=1)
axis.set_ylabel("Standardized value")
axis.set_title("Past-Only Context")
axis.legend()
finish_axis(axis)
save_figure(figure, "ipw_context_paths")

,Item,Value
0,Public source,Kenneth R. French 49 Industry Portfolios
1,Stored archive,49_Industry_Portfolios_daily_CSV.zip
2,Archive bytes,"4,173,817"
3,SHA256,8F394FE34BEA54D41B9AAFED410425EE8F8E252EDE3C71...
4,Analysis start,2019-01-02
5,Analysis end,2023-12-29
6,Complete dates,"1,258"
7,Download during execution,None


,Action,Basket,Industries,Daily return rule
0,0,Defensive,"Food, Smoke, Health, Utilities",Equal weighted
1,1,Cyclical,"Building Materials, Steel, Machinery, Automobiles",Equal weighted


,Component,Source or rule
0,Potential rewards,Both public basket returns on the same date
1,Context,Past returns only
2,Behavior action,Generated from a declared probability and rand...
3,Behavior probability floor,0.1
4,Target policies,Five fixed rules declared before held-out eval...


,Context,Dates,Mean,Standard deviation,Minimum,Maximum
0,Past mean spread,"1,258",0.00077,0.003047,-0.012438,0.010811
1,Past volatility,"1,258",0.012342,0.007947,0.003469,0.064707
2,Past one-day spread,"1,258",0.000776,0.013511,-0.0483,0.058375


,Quantity,Value
0,Minimum action probability,0.100043
1,Maximum action probability,0.899957
2,Mean probability of action 1,0.506933
3,Realized share of action 1,0.509539
4,Action 0 dates,617
5,Action 1 dates,641


,Policy,Probability of action 1
0,Always defensive,0
1,Always cyclical,1
2,Positive spread threshold,One when the past mean spread is positive
3,Smooth momentum,Logistic function of 1.5 times the past mean s...
4,Volatility aware rule,Logistic function of past spread and volatility


<Figure size 900x540 with 1 Axes>

<Figure size 900x540 with 1 Axes>

## Inverse Probability Value

For a target policy $pi(a\mid X_i)$ and behavior policy $\mu(a\mid X_i)$, the inverse probability estimate is

$$
\widehat V_{\mathrm{IPW}}(\pi)
=\frac{1}{n}\sum_{i=1}^{n}
\frac{\pi(A_i\mid X_i)}{\mu(A_i\mid X_i)}Y_i.
$$

Here, $n$ is the number of dates, $A_i$ is the logged action, $Y_i$ is its observed reward, and $X_i$ is the past-only context. The ratio gives more weight to a logged action when the target policy chooses it more often than the behavior policy did.

In [2]:
policy_rows = []
weight_rows = []
for name, policy in zip(names, candidates):
    result = evaluate_policy(actions, rewards, behavior, policy, np.zeros_like(behavior))
    truth = full_information_value(frame, policy)
    policy_rows.append({"Policy": name, "Full-information value": truth, "IPW value": result["ipw_value"], "Absolute error": abs(result["ipw_value"] - truth)})
    weight_rows.append({"Policy": name, "Effective sample size": result["effective_sample_size"], "Maximum weight": result["max_weight"], "Mean weight": result["mean_weight"]})
policy_values = pd.DataFrame(policy_rows)
weight_summary = pd.DataFrame(weight_rows)
save_table(policy_values, "ipw_policy_values", "Public full-information values and inverse probability estimates.", "tab:cim9-ipw-policy-values", fit=True)
save_table(weight_summary, "ipw_weight_summary", "Importance-weight diagnostics for the five target policies.", "tab:cim9-ipw-weight-summary", fit=True)

target = candidates[3]
target_result = evaluate_policy(actions, rewards, behavior, target, np.zeros_like(behavior))
row = np.arange(len(frame))
manual_rows = pd.DataFrame({
    "Date": frame.index[:8].date.astype(str),
    "Action": actions[:8],
    "Reward": rewards[:8],
    "Behavior probability": behavior[row[:8], actions[:8]],
    "Target probability": target[row[:8], actions[:8]],
    "Importance weight": target_result["importance_weights"][:8],
    "Weighted reward": target_result["ipw_scores"][:8],
})
save_table(manual_rows, "ipw_manual_score_rows", "First eight contributions for the smooth momentum policy.", "tab:cim9-ipw-manual-score-rows", fit=True)

support_checks = pd.DataFrame([
    {"Requirement": "Every target-supported action has positive behavior probability", "Status": bool(np.all(behavior[target > 0] > 0))},
    {"Requirement": "Every logged action has positive behavior probability", "Status": bool(np.all(behavior[row, actions] > 0))},
    {"Requirement": "All behavior rows sum to one", "Status": bool(np.allclose(behavior.sum(axis=1), 1))},
    {"Requirement": "All target rows sum to one", "Status": bool(np.allclose(target.sum(axis=1), 1))},
])
save_table(support_checks, "ipw_support_checks", "Overlap and probability checks for the main target policy.", "tab:cim9-ipw-support-checks", fit=True)

behavior_plot = frame[["Behavior probability action 1", "Action"]].reset_index()
behavior_plot.to_csv(FIGURE_DATA / "ipw_behavior_probabilities.csv", index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.4), constrained_layout=True)
axis.plot(frame.index, behavior[:, 1], color=PALETTE["cyan"], linewidth=1.2, label="Probability of action 1")
axis.scatter(frame.index[actions == 1], behavior[actions == 1, 1], color=PALETTE["amber"], s=8, alpha=0.45, label="Realized action 1")
axis.set_ylim(0, 1)
axis.set_ylabel("Probability")
axis.set_title("Generated Behavior Policy with Positive Support")
axis.legend()
finish_axis(axis)
save_figure(figure, "ipw_behavior_probabilities")

weight_plot = pd.DataFrame({"Importance weight": target_result["importance_weights"]})
weight_plot.to_csv(FIGURE_DATA / "ipw_weight_distribution.csv", index=False)
figure, axis = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
axis.hist(weight_plot["Importance weight"], bins=40, color=PALETTE["cyan"], alpha=0.82)
axis.axvline(target_result["mean_weight"], color=PALETTE["amber"], linewidth=2, linestyle="--", label="Mean weight")
axis.set_xlabel("Importance weight")
axis.set_ylabel("Dates")
axis.set_title("Importance Weights for Smooth Momentum")
axis.legend()
finish_axis(axis)
save_figure(figure, "ipw_weight_distribution")

,Policy,Full-information value,IPW value,Absolute error
0,Always defensive,0.000394,0.000269,0.000125
1,Always cyclical,0.001163,-0.00029,0.001454
2,Positive spread threshold,0.000727,0.000318,0.000408
3,Smooth momentum,0.000845,0.000444,0.000401
4,Volatility aware rule,0.000684,0.000719,3.50379e-05


,Policy,Effective sample size,Maximum weight,Mean weight
0,Always defensive,522.476575,6.553751,1.007221
1,Always cyclical,519.419969,9.983498,1.017048
2,Positive spread threshold,699.620353,6.920637,0.982218
3,Smooth momentum,1019.217097,6.118852,0.98246
4,Volatility aware rule,1167.543731,1.524121,0.993597


,Date,Action,Reward,Behavior probability,Target probability,Importance weight,Weighted reward
0,2019-01-02,1,0.004275,0.388654,0.69201,1.780531,0.007612
1,2019-01-03,1,-0.02535,0.417775,0.663136,1.587306,-0.040238
2,2019-01-04,0,0.0269,0.720187,0.527095,0.731885,0.019688
3,2019-01-07,1,0.012825,0.485841,0.800249,1.647141,0.021125
4,2019-01-08,0,0.012075,0.47175,0.118607,0.251418,0.003036
5,2019-01-09,0,0.00345,0.472552,0.072337,0.153077,0.000528
6,2019-01-10,1,0.00805,0.596548,0.952078,1.595978,0.012848
7,2019-01-11,0,0.0026,0.405666,0.03372,0.083123,0.000216


,Requirement,Status
0,Every target-supported action has positive beh...,Pass
1,Every logged action has positive behavior prob...,Pass
2,All behavior rows sum to one,Pass
3,All target rows sum to one,Pass


<Figure size 900x540 with 1 Axes>

<Figure size 880x520 with 1 Axes>

## Diagnosing weight variation under limited overlap

Overlap requires

$$
\mu(a\mid x)>0
\quad\text{whenever}\quad
\pi(a\mid x)>0.
$$

The condition says that a target action must have some chance of appearing under the behavior policy at every relevant context $x$. It permits inverse probability evaluation, but it does not promise small weights.

A useful summary of weight concentration is

$$
n_{\mathrm{eff}}
=\frac{\left(\sum_{i=1}^{n}w_i\right)^2}{\sum_{i=1}^{n}w_i^2},
\qquad
w_i=\frac{\pi(A_i\mid X_i)}{\mu(A_i\mid X_i)}.
$$

Here, $w_i$ is date $i$'s importance weight and $n_{\mathrm{eff}}$ is its effective sample size. A small value indicates that a few dates carry much of the weighted calculation.

In [3]:
overlap_rows = []
for floor in [0.01, 0.03, 0.05, 0.10, 0.20]:
    alternative = build_public_frame(floor=floor)
    alternative_behavior = behavior_matrix(alternative)
    alternative_actions = alternative["Action"].to_numpy()
    alternative_rewards = alternative["Observed reward"].to_numpy()
    alternative_target = candidate_policies(alternative)[1][3]
    result = evaluate_policy(alternative_actions, alternative_rewards, alternative_behavior, alternative_target, np.zeros_like(alternative_behavior))
    truth = full_information_value(alternative, alternative_target)
    overlap_rows.append({
        "Probability floor": floor,
        "Minimum action probability": alternative_behavior.min(),
        "Realized action 1 share": alternative_actions.mean(),
        "Effective sample size": result["effective_sample_size"],
        "Maximum weight": result["max_weight"],
        "IPW value": result["ipw_value"],
        "Full-information value": truth,
        "Absolute error": abs(result["ipw_value"] - truth),
    })
overlap_sensitivity = pd.DataFrame(overlap_rows)
save_table(overlap_sensitivity, "ipw_overlap_sensitivity", "Sensitivity of the smooth momentum estimate to the behavior probability floor.", "tab:cim9-ipw-overlap-sensitivity", fit=True)

invalid_behavior = behavior.copy()
invalid_behavior[0] = [1, 0]
invalid_policy = target.copy()
invalid_policy[0] = [0, 1]
invalid_checks = pd.DataFrame([
    capture_failure("Target action without behavior support", lambda: evaluate_policy(actions, rewards, invalid_behavior, invalid_policy, predictions)),
    capture_failure("Probability rows not summing to one", lambda: evaluate_policy(actions, rewards, behavior * 0.9, target, predictions)),
])
save_table(invalid_checks, "ipw_invalid_support_checks", "Required rejections for invalid probability support.", "tab:cim9-ipw-invalid-support-checks", fit=True)

section_summary = pd.DataFrame([
    {"Result": "Public reward information", "Value": "Both basket returns are observed for every date"},
    {"Result": "Generated component", "Value": "Only the behavior action log and its probabilities"},
    {"Result": "Behavior support", "Value": "Every action probability is bounded away from zero"},
    {"Result": "IPW interpretation", "Value": "Average weighted reward for a fixed one-step target policy"},
    {"Result": "Excluded interpretation", "Value": "No sequential reinforcement learning or live performance claim"},
])
save_table(section_summary, "ipw_section_summary", "Summary of the inverse probability illustration.", "tab:cim9-ipw-section-summary", fit=True)

overlap_sensitivity.to_csv(FIGURE_DATA / "ipw_overlap_sensitivity.csv", index=False)
figure, left = plt.subplots(figsize=(8.8, 5.4), constrained_layout=True)
right = left.twinx()
left.plot(overlap_sensitivity["Probability floor"], overlap_sensitivity["Effective sample size"], color=PALETTE["cyan"], marker="o", linewidth=2.2, label="Effective sample size")
right.plot(overlap_sensitivity["Probability floor"], overlap_sensitivity["Maximum weight"], color=PALETTE["amber"], marker="s", linewidth=2.2, label="Maximum weight")
left.set_xlabel("Behavior probability floor")
left.set_ylabel("Effective sample size", color=PALETTE["cyan"])
right.set_ylabel("Maximum importance weight", color=PALETTE["amber"])
left.set_title("Overlap Changes Weight Concentration")
left.grid(True, alpha=0.65)
lines = left.get_lines() + right.get_lines()
left.legend(lines, [line.get_label() for line in lines], loc="center right")
save_figure(figure, "ipw_overlap_sensitivity")

,Probability floor,Minimum action probability,Realized action 1 share,Effective sample size,Maximum weight,IPW value,Full-information value,Absolute error
0,0.01,0.010052,0.509539,937.656403,13.706234,0.000362,0.000845,0.000483
1,0.03,0.03005,0.509539,978.658807,10.745303,0.000458,0.000845,0.000387
2,0.05,0.050048,0.505564,997.375061,8.836395,0.000458,0.000845,0.000386
3,0.1,0.100043,0.509539,1019.217097,6.118852,0.000444,0.000845,0.000401
4,0.2,0.200032,0.503975,1015.08698,3.788576,0.000498,0.000845,0.000347


,Check,Rejected,Explanation
0,Target action without behavior support,Pass,Stopped as required
1,Probability rows not summing to one,Pass,Stopped as required


,Result,Value
0,Public reward information,Both basket returns are observed for every date
1,Generated component,Only the behavior action log and its probabili...
2,Behavior support,Every action probability is bounded away from ...
3,IPW interpretation,Average weighted reward for a fixed one-step t...
4,Excluded interpretation,No sequential reinforcement learning or live p...


<Figure size 880x540 with 2 Axes>